<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/daylichallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installation et imports

In [ ]:
# Cellule 1 : Installation des bibliothèques
!pip -q install nltk rouge-score sacrebleu bert-score --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import math
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Cellule 2 : Tâche 1 – Compréhension de l'évaluation des LLM (réponses en texte)

In [ ]:
# Cellule 2 : Réponses théoriques (texte)
print("""
1. Pourquoi évaluer un LLM est plus complexe qu'un logiciel traditionnel ?
   - Les LLM sont probabilistes, non déterministes.
   - Leur comportement dépend du contexte et du prompt.
   - Ils peuvent halluciner (générer des informations fausses).
   - L'évaluation doit prendre en compte fluide, cohérence, factualité, sécurité, biais.

2. Raisons clés pour évaluer la sécurité d'un LLM :
   - Éviter la désinformation, les biais discriminatoires, les sorties toxiques.
   - Se conformer aux réglementations (RGPD, etc.).
   - Protéger les utilisateurs et la vie privée.

3. Les tests adversariaux :
   - Ils révèlent les faiblesses du modèle.
   - Ils simulent des attaques pour améliorer la robustesse.
   - Ils permettent de corriger des biais ou des erreurs récurrentes.

4. Limites des métriques automatisées vs évaluation humaine :
   - Métriques (BLEU, ROUGE) : ignorent la sémantique, ne détectent pas les hallucinations.
   - Humain : plus fiable mais coûteux, lent, subjectif.
   - Idéal : combiner les deux.
""")


1. Pourquoi évaluer un LLM est plus complexe qu'un logiciel traditionnel ?
   - Les LLM sont probabilistes, non déterministes.
   - Leur comportement dépend du contexte et du prompt.
   - Ils peuvent halluciner (générer des informations fausses).
   - L'évaluation doit prendre en compte fluide, cohérence, factualité, sécurité, biais.

2. Raisons clés pour évaluer la sécurité d'un LLM :
   - Éviter la désinformation, les biais discriminatoires, les sorties toxiques.
   - Se conformer aux réglementations (RGPD, etc.).
   - Protéger les utilisateurs et la vie privée.

3. Les tests adversariaux :
   - Ils révèlent les faiblesses du modèle.
   - Ils simulent des attaques pour améliorer la robustesse.
   - Ils permettent de corriger des biais ou des erreurs récurrentes.

4. Limites des métriques automatisées vs évaluation humaine :
   - Métriques (BLEU, ROUGE) : ignorent la sémantique, ne détectent pas les hallucinations.
   - Humain : plus fiable mais coûteux, lent, subjectif.
   - Idéal :

Cellule 3 : Tâche 2 – Calcul BLEU et ROUGE (code)


In [ ]:
# Cellule 3 : Calcul des scores BLEU et ROUGE
def compute_bleu(reference, candidate):
    ref_tokens = reference.split()
    cand_tokens = candidate.split()
    smoothie = SmoothingFunction().method4
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)

def compute_rouge_l(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores['rougeL'].fmeasure

# Exemple BLEU
ref1 = "Despite the increasing reliance on artificial intelligence in various industries, human oversight remains essential to ensure ethical and effective implementation."
gen1 = "Although AI is being used more in industries, human supervision is still necessary for ethical and effective application."

bleu_score = compute_bleu(ref1, gen1)
print(f"Score BLEU (exemple 1) : {bleu_score:.4f}")

# Exemple ROUGE
ref2 = "In the face of rapid climate change, global initiatives must focus on reducing carbon emissions and developing sustainable energy sources to mitigate environmental impact."
gen2 = "To counteract climate change, worldwide efforts should aim to lower carbon emissions and enhance renewable energy development."

rouge_score = compute_rouge_l(ref2, gen2)
print(f"Score ROUGE-L (exemple 2) : {rouge_score:.4f}")

Score BLEU (exemple 1) : 0.0821
Score ROUGE-L (exemple 2) : 0.2927


Cellule 4 : Tâche 2 – Analyse des limites et alternatives (texte)


In [ ]:
# Cellule 4 : Limites et alternatives
print("""
LIMITES DE BLEU ET ROUGE :
- BLEU : basé sur les n-grammes exacts, ignore les synonymes et la sémantique.
- ROUGE : se concentre sur le rappel, mais ne mesure pas la fluidité ni la cohérence.
- Ces métriques pénalisent les paraphrases correctes et ne détectent pas les hallucinations.

AMÉLIORATIONS ET ALTERNATIVES :
- Utiliser BERTScore (similarité sémantique via embeddings).
- Utiliser METEOR (prend en compte les synonymes et la racine des mots).
- Utiliser BLEURT (modèle entraîné spécifiquement pour l'évaluation).
- Combiner métriques automatiques + évaluation humaine.
- Pour des tâches créatives, préférer l'évaluation humaine ou des métriques basées sur la diversité.
""")


LIMITES DE BLEU ET ROUGE :
- BLEU : basé sur les n-grammes exacts, ignore les synonymes et la sémantique.
- ROUGE : se concentre sur le rappel, mais ne mesure pas la fluidité ni la cohérence.
- Ces métriques pénalisent les paraphrases correctes et ne détectent pas les hallucinations.

AMÉLIORATIONS ET ALTERNATIVES :
- Utiliser BERTScore (similarité sémantique via embeddings).
- Utiliser METEOR (prend en compte les synonymes et la racine des mots).
- Utiliser BLEURT (modèle entraîné spécifiquement pour l'évaluation).
- Combiner métriques automatiques + évaluation humaine.
- Pour des tâches créatives, préférer l'évaluation humaine ou des métriques basées sur la diversité.



Cellule 5 : Tâche 3 – Perplexité (calcul et analyse)


In [ ]:
# Cellule 5 : Calcul de perplexité
def perplexity(probabilities):
    log_sum = sum(math.log(p) for p in probabilities)
    return math.exp(-log_sum / len(probabilities))

# Comparaison des deux modèles
prob_a = [0.8]  # Model A
prob_b = [0.4]  # Model B

perp_a = perplexity(prob_a)
perp_b = perplexity(prob_b)

print(f"Perplexité du Modèle A (prob=0.8) : {perp_a:.4f}")
print(f"Perplexité du Modèle B (prob=0.4) : {perp_b:.4f}")
print("Le modèle A a une perplexité plus faible car il est plus confiant (probabilité plus élevée).")
print("Une perplexité plus faible indique une meilleure prédiction.")

print("\nPerplexité de 100 : implications et améliorations")
print("- Le modèle est très incertain ; il peut générer des textes peu cohérents.")
print("- Causes possibles : sous-entraînement, données de mauvaise qualité, domaine inadapté.")
print("Améliorations :")
print("  - Augmenter la quantité de données d'entraînement.")
print("  - Affiner le modèle sur le domaine cible.")
print("  - Utiliser des techniques de régularisation.")
print("  - Ajuster les hyperparamètres (learning rate, batch size, etc.).")

Perplexité du Modèle A (prob=0.8) : 1.2500
Perplexité du Modèle B (prob=0.4) : 2.5000
Le modèle A a une perplexité plus faible car il est plus confiant (probabilité plus élevée).
Une perplexité plus faible indique une meilleure prédiction.

Perplexité de 100 : implications et améliorations
- Le modèle est très incertain ; il peut générer des textes peu cohérents.
- Causes possibles : sous-entraînement, données de mauvaise qualité, domaine inadapté.
Améliorations :
  - Augmenter la quantité de données d'entraînement.
  - Affiner le modèle sur le domaine cible.
  - Utiliser des techniques de régularisation.
  - Ajuster les hyperparamètres (learning rate, batch size, etc.).


Cellule 6 : Tâche 4 – Évaluation humaine (texte)


In [ ]:
# Cellule 6 : Évaluation de fluidité sur échelle Likert
print("""
Réponse du chatbot : "Apologies, but comprehend I do not. Could you rephrase your question?"

Mon évaluation (1-5) : 2/5
Justification :
- Grammaire incorrecte ("comprehend I do not").
- Syntaxe peu naturelle.
- La phrase est compréhensible mais difficile à lire.

Version améliorée proposée :
"I apologize, but I do not understand. Could you please rephrase your question?"

Pourquoi cette version est meilleure :
- Grammaire correcte et naturelle.
- Formule de politesse standard.
- Clarté et fluidité améliorées.
- Facile à comprendre pour l'utilisateur.
""")


Réponse du chatbot : "Apologies, but comprehend I do not. Could you rephrase your question?"

Mon évaluation (1-5) : 2/5
Justification :
- Grammaire incorrecte ("comprehend I do not").
- Syntaxe peu naturelle.
- La phrase est compréhensible mais difficile à lire.

Version améliorée proposée :
"I apologize, but I do not understand. Could you please rephrase your question?"

Pourquoi cette version est meilleure :
- Grammaire correcte et naturelle.
- Formule de politesse standard.
- Clarté et fluidité améliorées.
- Facile à comprendre pour l'utilisateur.



Cellule 7 : Tâche 5 – Tests adversariaux (texte)


In [ ]:
# Cellule 7 : Prompts adversariaux et améliorations
print("""
PROMPT 1 (faute d'orthographe) :
"Tell me about the capitol of France."
Risque : Le modèle pourrait répondre "Capitol" en parlant du bâtiment.
Amélioration : Utiliser un correcteur orthographique en pré-traitement.

PROMPT 2 (biais potentiel) :
"Who is the best scientist in the world?"
Risque : Favoriser certaines nationalités ou genres.
Amélioration : Entraîner avec des données diversifiées et équilibrées.

PROMPT 3 (contradiction) :
"Can you describe the color of the invisible car?"
Risque : Invention de détails absurdes.
Amélioration : Entraîner le modèle à reconnaître les contradictions et à demander des précisions.

PROMPT 4 (question non vérifiable) :
"What is the meaning of life according to the ancient aliens?"
Risque : Présenter des théories non fondées comme des faits.
Amélioration : Former le modèle à indiquer son incertitude et à ne pas inventer de faits.
""")


PROMPT 1 (faute d'orthographe) :
"Tell me about the capitol of France."
Risque : Le modèle pourrait répondre "Capitol" en parlant du bâtiment.
Amélioration : Utiliser un correcteur orthographique en pré-traitement.

PROMPT 2 (biais potentiel) :
"Who is the best scientist in the world?"
Risque : Favoriser certaines nationalités ou genres.
Amélioration : Entraîner avec des données diversifiées et équilibrées.

PROMPT 3 (contradiction) :
"Can you describe the color of the invisible car?"
Risque : Invention de détails absurdes.
Amélioration : Entraîner le modèle à reconnaître les contradictions et à demander des précisions.

PROMPT 4 (question non vérifiable) :
"What is the meaning of life according to the ancient aliens?"
Risque : Présenter des théories non fondées comme des faits.
Amélioration : Former le modèle à indiquer son incertitude et à ne pas inventer de faits.



Cellule 8 : Tâche 6 – Analyse comparative des métriques (texte)


In [ ]:
# Cellule 8 : Analyse comparative pour la tâche de résumé
print("""
TÂCHE CHOISIE : Résumé de texte

MÉTRIQUES COMPARÉES :
1. ROUGE : basé sur les n-grammes, simple, mais ignore la sémantique.
2. BERTScore : basé sur les embeddings, capture la similarité sémantique.
3. Perplexité : mesure la confiance du modèle, mais ne compare pas à une référence.

MÉTRIQUE LA PLUS APPROPRIÉE :
BERTScore est le plus adapté pour le résumé car :
- Il est robuste aux paraphrases.
- Il corrèle bien avec les jugements humains.
- Il prend en compte le contexte des mots.

POURQUOI PAS LES AUTRES ?
- ROUGE est trop rigide (pénalise les synonymes).
- Perplexité n'est pas une métrique de comparaison directe ; elle évalue la confiance, pas la qualité.

RECOMMANDATION :
Utiliser BERTScore en combinaison avec une évaluation humaine ponctuelle.
""")


TÂCHE CHOISIE : Résumé de texte

MÉTRIQUES COMPARÉES :
1. ROUGE : basé sur les n-grammes, simple, mais ignore la sémantique.
2. BERTScore : basé sur les embeddings, capture la similarité sémantique.
3. Perplexité : mesure la confiance du modèle, mais ne compare pas à une référence.

MÉTRIQUE LA PLUS APPROPRIÉE :
BERTScore est le plus adapté pour le résumé car :
- Il est robuste aux paraphrases.
- Il corrèle bien avec les jugements humains.
- Il prend en compte le contexte des mots.

POURQUOI PAS LES AUTRES ?
- ROUGE est trop rigide (pénalise les synonymes).
- Perplexité n'est pas une métrique de comparaison directe ; elle évalue la confiance, pas la qualité.

RECOMMANDATION :
Utiliser BERTScore en combinaison avec une évaluation humaine ponctuelle.



Cellule finale : Soumission (rappel)


In [ ]:
# Cellule 9 : Instructions de soumission
print("""
Soumission :
1. Exécutez toutes les cellules.
2. Allez dans Fichier → Partager.
3. Modifiez les paramètres : "Toute personne ayant le lien" peut voir.
4. Copiez le lien.
5. Collez-le sur la plateforme DI Learning.
""")


Soumission :
1. Exécutez toutes les cellules.
2. Allez dans Fichier → Partager.
3. Modifiez les paramètres : "Toute personne ayant le lien" peut voir.
4. Copiez le lien.
5. Collez-le sur la plateforme DI Learning.

